# Day 12 Revision Summary — SQL, Datasets & Splits

- **SQL mirrors pandas**: `SELECT`/`WHERE` ≈ column selection + boolean filter, `GROUP BY` ≈ `df.groupby(...).agg(...)`, `JOIN` ≈ `pd.merge(...)` — same operations, different keywords, and SQLite (`sqlite3`) is built into Python so no server setup is needed.
- `pd.read_sql(query, conn)` lets you query a database and get a DataFrame straight back — the database does the fast filtering/joining, and pandas takes over for analysis and modelling.
- **Training data comes from several places**: your own CSV/SQL data, built-in scikit-learn datasets (`iris`, `wine`, `digits`), and HuggingFace's `load_dataset(...)` (the go-to source from Week 6 onward).
- **The golden rule of ML**: never test a model on data it trained on — hold out a test set the model never sees, so its reported score can actually be trusted.
- `train_test_split(X, y, test_size=..., random_state=42, stratify=y)` builds reproducible, class-balanced train/validation/test splits; the test set should be looked at **once**, at the very end.

*No shared "formulas" resource file was referenced by Day 12's material; only Day 10 pointed at a shared resource (`resources/week2_revision_sheet.md`).*


## Classwork Exercise 1 — Query a database (`sql_intro.py`, ~15 min)

**What's being asked:** Load `students.csv` and `scores.csv` into a SQLite database, then answer three questions in SQL: all Math scores above 60, the average score and count per subject (`GROUP BY`), and the average score per city (`JOIN` + `GROUP BY`) — then compare each SQL result against the equivalent pandas operation.

**Approach:**
1. `import sqlite3` and `pandas as pd`; read `students.csv` and `scores.csv` into DataFrames with `pd.read_csv`.
2. Open a connection: `conn = sqlite3.connect("school.db")`; write both DataFrames into it with `df.to_sql("students", conn, if_exists="replace", index=False)` (and similarly for `scores`).
3. Run `pd.read_sql("SELECT * FROM scores WHERE subject='Math' AND score > 60", conn)` and print it.
4. Run a `GROUP BY` query: `SELECT subject, AVG(score) AS avg_score, COUNT(*) AS n FROM scores GROUP BY subject`.
5. Run a `JOIN` + `GROUP BY` query joining `students` and `scores` on `student_id`, grouped by `city`, with `AVG(score)`.
6. For each SQL query, also compute the same answer directly in pandas (filter / `groupby` / `merge`+`groupby`) and confirm the numbers match.


In [ ]:
# Day 12 · Classwork Exercise 1 — Query a database
# Source: sql_intro.py

import sqlite3

import pandas as pd

# TODO: load the CSVs (adjust filenames/columns to whatever's provided)
# students = pd.read_csv("students.csv")   # e.g. columns: student_id, name, city
# scores = pd.read_csv("scores.csv")       # e.g. columns: student_id, subject, score

conn = sqlite3.connect("school.db")

# TODO: write both DataFrames into the SQLite DB
# students.to_sql("students", conn, if_exists="replace", index=False)
# scores.to_sql("scores", conn, if_exists="replace", index=False)

# TODO: Query 1 -- all Math scores above 60
# q1 = pd.read_sql("SELECT * FROM scores WHERE subject='Math' AND score > 60", conn)
# print(q1)

# TODO: Query 2 -- average score and count per subject
# q2 = pd.read_sql('''
#     SELECT subject, AVG(score) AS avg_score, COUNT(*) AS n
#     FROM scores
#     GROUP BY subject
# ''', conn)
# print(q2)

# TODO: Query 3 -- average score per city (JOIN + GROUP BY)
# q3 = pd.read_sql('''
#     SELECT st.city, AVG(sc.score) AS avg_score
#     FROM students st
#     JOIN scores sc ON st.student_id = sc.student_id
#     GROUP BY st.city
# ''', conn)
# print(q3)

# TODO: cross-check each query against the equivalent pandas filter/groupby/merge


## Classwork Exercise 2 — Split it right (`splits.py`, ~15 min)

**What's being asked:** Load the built-in `iris` dataset, split it 80/20 with a fixed `random_state`, confirm the split is reproducible across runs, and confirm that adding `stratify=y` keeps the three iris classes balanced across the split.

**Approach:**
1. `from sklearn.datasets import load_iris` and `from sklearn.model_selection import train_test_split`.
2. `iris = load_iris(); X, y = iris.data, iris.target`.
3. `X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)`; print all four shapes.
4. Run the split again with the same `random_state` and confirm (e.g. via `np.array_equal`) that `X_train` is identical both times.
5. Re-run with `stratify=y` added; use `np.bincount(y_train)` / `np.bincount(y_test)` (or `pd.Series(y_train).value_counts()`) to confirm each class keeps roughly its original proportion.


In [ ]:
# Day 12 · Classwork Exercise 2 — Split it right
# Source: splits.py

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target

# TODO: 80/20 split with a fixed random_state
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO: print the four shapes
# print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# TODO: run the split again with the same random_state and confirm it's identical
# X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, test_size=0.2, random_state=42)
# print(np.array_equal(X_train, X_train2))  # expect: True

# TODO: re-run with stratify=y and check class balance
# X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )
# print(np.bincount(y_train_s), np.bincount(y_test_s))


## Homework Exercise 3 — Three more SQL queries (~homework)

**What's being asked:** Write three additional SQL queries against the school database: one using `HAVING` to filter on an aggregate, and one combining `JOIN` with `GROUP BY` (distinct from the Exercise 1 query).

**Approach:**
1. Reconnect to `school.db` (reuse Exercise 1's setup).
2. Write a query with `GROUP BY ... HAVING AVG(score) > <threshold>` (or `HAVING COUNT(*) > <n>`) to filter **groups**, not rows.
3. Write a second `JOIN` + `GROUP BY` query answering a new question (e.g. average score per subject per city, or the city with the most passing students).
4. Write a third query of your choice reusing `SELECT`/`WHERE`/`ORDER BY`/`LIMIT`.
5. Run all three with `pd.read_sql(...)` and print the resulting DataFrames.


In [ ]:
# Day 12 · Homework Exercise 3 — Three more SQL queries

import sqlite3

import pandas as pd

conn = sqlite3.connect("school.db")

# TODO: Query A -- GROUP BY ... HAVING (filter on an aggregate, not a row)
# qa = pd.read_sql('''
#     SELECT subject, AVG(score) AS avg_score
#     FROM scores
#     GROUP BY subject
#     HAVING AVG(score) > 70
# ''', conn)
# print(qa)

# TODO: Query B -- a different JOIN + GROUP BY question
# qb = pd.read_sql('''
#     SELECT ...
#     FROM students st
#     JOIN scores sc ON st.student_id = sc.student_id
#     GROUP BY ...
# ''', conn)
# print(qb)

# TODO: Query C -- your choice, using SELECT/WHERE/ORDER BY/LIMIT
# qc = pd.read_sql("SELECT ... ORDER BY ... LIMIT ...", conn)
# print(qc)


## Homework Exercise 4 — 60/20/20 stratified split (~homework)

**What's being asked:** Split the iris dataset into train/validation/test sets (60/20/20) using two calls to `train_test_split` with `stratify`, and print the shapes of all six resulting arrays.

**Approach:**
1. First split off the test set: `X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`.
2. Second split divides the remaining 80% into 60% train / 20% val, i.e. `test_size=0.25` of the tmp set: `X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp)`.
3. Print `X_train.shape, X_val.shape, X_test.shape, y_train.shape, y_val.shape, y_test.shape`.
4. Confirm the proportions roughly match 60/20/20 of the full dataset.


In [ ]:
# Day 12 · Homework Exercise 4 — 60/20/20 stratified split

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target

# TODO: first split -- 80% tmp / 20% test, stratified
# X_tmp, X_test, y_tmp, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )

# TODO: second split -- 60% train / 20% val out of the tmp set (test_size=0.25 of tmp), stratified
# X_train, X_val, y_train, y_val = train_test_split(
#     X_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp
# )

# TODO: print all six shapes
# print(X_train.shape, X_val.shape, X_test.shape, y_train.shape, y_val.shape, y_test.shape)

# TODO: sanity-check the proportions are roughly 60/20/20 of the full dataset


## Other homework items (no code needed)

- In one sentence each, answer: why set `random_state`? Why should you never peek at the test set before the final evaluation?
- Commit your work: `git add . && git commit -m "day 12"`.
